# 推荐系统 Recommendation Systems

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/logo.png" width=150>

推荐系统用于向用户推荐可能感兴趣的内容。本notebook介绍协同过滤和矩阵分解等经典推荐方法。

Recommendation systems are used to recommend content that users might be interested in. This notebook introduces classic recommendation methods like collaborative filtering and matrix factorization.

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/recommend.png" width=500>

# 概述 Overview

* **目标:**  预测用户对物品的偏好，向用户推荐新物品。
* **优点:** 
  * 提升用户体验
  * 增加平台收益
  * 适用于各种场景
* **缺点:**
  * 冷启动问题
  * 数据稀疏性
* **其他:** 
  * 深度学习推荐系统 (DNN)
  * 基于内容的推荐

# 设置 Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

# 用户-物品交互矩阵 User-Item Interaction Matrix

In [ ]:
# 示例评分数据 Example rating data
# 用户 1-5 对物品 A-E 的评分
ratings_data = {
    'user_id': [1, 1, 1, 2, 2, 3, 3, 3, 4, 4, 5, 5],
    'item_id': ['A', 'B', 'C', 'A', 'D', 'B', 'C', 'E', 'A', 'E', 'D', 'E'],
    'rating': [5, 4, 3, 4, 5, 3, 4, 5, 3, 4, 5, 4]
}

ratings_df = pd.DataFrame(ratings_data)
print("Ratings DataFrame:")
print(ratings_df)

In [ ]:
# 创建用户-物品矩阵 Create user-item matrix
user_item_matrix = ratings_df.pivot_table(
    index='user_id',
    columns='item_id',
    values='rating'
).fillna(0)

print("\nUser-Item Matrix:")
print(user_item_matrix)

# 协同过滤 Collaborative Filtering

In [ ]:
# 计算用户相似度 Compute user similarity
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

print("User Similarity Matrix:")
print(user_similarity_df)

In [ ]:
# 基于用户的协同过滤预测 User-based CF prediction
def predict_user_based(user_id, item_id, user_item_matrix, user_similarity_df):
    # 获取该用户对所有物品的评分
    user_ratings = user_item_matrix.loc[user_id]
    
    # 找到对该物品评分的其他用户
    other_users = user_item_matrix[user_item_matrix[item_id] > 0].index
    other_users = other_users[other_users != user_id]
    
    if len(other_users) == 0:
        return 0
    
    # 加权平均 Weighted average
    numerator = 0
    denominator = 0
    for other_user in other_users:
        sim = user_similarity_df.loc[user_id, other_user]
        rating = user_item_matrix.loc[other_user, item_id]
        numerator += sim * rating
        denominator += abs(sim)
    
    return numerator / denominator if denominator > 0 else 0

# 预测用户 1 对物品 E 的评分
prediction = predict_user_based(1, 'E', user_item_matrix, user_similarity_df)
print(f"Predicted rating for User 1 on Item E: {prediction:.2f}")

# 矩阵分解 Matrix Factorization

In [ ]:
# 简化的矩阵分解 Simplified Matrix Factorization
class MatrixFactorization:
    def __init__(self, n_factors=2, learning_rate=0.01, n_epochs=100):
        self.n_factors = n_factors
        self.lr = learning_rate
        self.n_epochs = n_epochs
    
    def fit(self, user_item_matrix):
        self.n_users, self.n_items = user_item_matrix.shape
        
        # 初始化用户和物品嵌入 Initialize user and item embeddings
        np.random.seed(42)
        self.user_factors = np.random.randn(self.n_users, self.n_factors)
        self.item_factors = np.random.randn(self.n_items, self.n_factors)
        
        # 训练 Training
        for epoch in range(self.n_epochs):
            for u in range(self.n_users):
                for i in range(self.n_items):
                    if user_item_matrix.iloc[u, i] > 0:
                        error = user_item_matrix.iloc[u, i] - \
                                np.dot(self.user_factors[u], self.item_factors[i])
                        self.user_factors[u] += self.lr * error * self.item_factors[i]
                        self.item_factors[i] += self.lr * error * self.user_factors[u]
    
    def predict(self, user_id, item_id):
        user_idx = list(user_item_matrix.index).index(user_id)
        item_idx = list(user_item_matrix.columns).index(item_id)
        return np.dot(self.user_factors[user_idx], self.item_factors[item_idx])

# 训练模型
mf = MatrixFactorization(n_factors=2, learning_rate=0.01, n_epochs=100)
mf.fit(user_item_matrix)

# 预测
prediction = mf.predict(1, 'E')
print(f"MF Predicted rating for User 1 on Item E: {prediction:.2f}")

# TODO

- 深度协同过滤 Deep Collaborative Filtering
- 基于内容的推荐 Content-Based Recommendations
- 序列推荐 Sequence Recommendation
- 推荐系统评估指标 Evaluation Metrics (Hit Rate, NDCG)